<a href="https://colab.research.google.com/github/johnsondillond/NLPProject/blob/main/CQADataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing of CQA Data

## Setup of Environment

In [1]:
!if [[ ! -d NLPProject/ ]]; then git clone https://github.com/johnsondillond/NLPProject.git; fi

In [2]:
import polars as pl

In [3]:
# Set up different data file names with CQA names & target classes
cqa_names = ['AskUbuntu', 'FaultServer', 'Mathematics', 'MetaStackExchange', 'SuperUser']
post_classes = ['HQ', 'LQE', 'LQC']

## Analyze Dataset Shape and Distribution on Original Data and CQA Data

In [4]:
data_dirpath = "NLPProject/CQA_Data"
df_list = []
for cqa in cqa_names:
  for post_class in post_classes:
    df_list.append(pl.read_csv(f"{data_dirpath}/{cqa}{post_class}.csv"))

In [5]:
hq_samples, lqe_samples, lqc_samples, total_samples = 0, 0, 0, 0
for idx, df in enumerate(df_list):
  if idx % 3 == 0:
    hq_samples += df.shape[0]
  if idx % 3 == 1:
    lqe_samples += df.shape[0]
  else:
    lqc_samples += df.shape[0]

  # Assign Y (target variable based on proper df from csv)
  df.insert_column(5, pl.Series('Y', [post_classes[idx%3]]*df.shape[0]))
  # Assign CQA as a feature to understand which CQA is predicted most accurately
  df.insert_column(6, pl.Series('CQA', [cqa_names[idx//3]]*df.shape[0]))
  total_samples += df.shape[0]
print(f"Total Samples in CQA_Data: {total_samples}")
print(f"HQ Sample Distribution: {hq_samples / total_samples}\n\
LQE Sample Distribution: {lqe_samples / total_samples}\n\
LQC Sample Distribution: {lqc_samples / total_samples}")


Total Samples in CQA_Data: 244450
HQ Sample Distribution: 0.8299734096952343
LQE Sample Distribution: 0.03217426876661894
LQC Sample Distribution: 0.9678257312333811


In [6]:
original_dataset_path = 'NLPProject/data/total.csv'
original_df = pl.read_csv(original_dataset_path)
original_df['Y'].value_counts()

['Id', 'Title', 'Body', 'Tags', 'CreationDate', 'Y']


Y,count
str,u32
"""LQ_EDIT""",20000
"""LQ_CLOSE""",20000
"""HQ""",20000


## Normalize Class Distribution Aggregation

In [7]:
norm_data_size = lqe_samples * 3
per_class_sample_size = lqe_samples // 5
print(f'Test Dataset Size: {norm_data_size}\n\
Each class will provide {per_class_sample_size} samples')

Test Dataset Size: 23595
Each class will provide 1573 samples


In [8]:
normalized_df_list = []
missing_samples = 0
for idx, df in enumerate(df_list):
  # Skip LQE since all samples are being used from them
  if idx % 3 == 1:
    normalized_df_list.append(df)
    continue
  if df.shape[0] >= per_class_sample_size:
    # MetaStackExchange has a sample that is too small, so SuperUser will take on the remaining 145
    if idx % 3 == 2 and missing_samples > 0:
      normalized_df_list.append(df.sample(per_class_sample_size+missing_samples, seed=123))
      print(f"{df[1, 6]} is taking an additional {missing_samples} samples to give an equal class distribution")
    else:
      normalized_df_list.append(df.sample(per_class_sample_size, seed=123))
  else:
    normalized_df_list.append(df.sample(fraction=1.0, seed=123))
    missing_samples += per_class_sample_size - df.shape[0]
    print(f"{df[1, 6]} has {missing_samples} too few of samples")

MetaStackExchange has 145 too few of samples
SuperUser is taking an additional 145 samples to give an equal class distribution


## Concatenate Normalized DataSets

In [11]:
# Set the test set to a shuffled version of the concatenated dataframes
test_set = pl.concat(
    normalized_df_list
).sample(fraction=1.0, seed=213, shuffle=True)
print(test_set.shape)
# Test Dataset Size should be equal to normalized dataset size determined earlier
assert test_set.shape[0] == norm_data_size
# Verify Target Variables have been shuffled
test_set['Y'].value_counts()

(23595, 7)


Y,count
str,u32
"""LQE""",7865
"""HQ""",7865
"""LQC""",7865


In [10]:
test_set.write_csv(f"{data_dirpath}/test.csv")